# 🔍 HarMeme OCR — Notebook 2: Chunk Processor
Runs Tesseract OCR on one chunk of images and saves results as a `.pkl` file.

**For each chunk:**
1. Set `CHUNK_ID = 0` (or 1, 2, 3)
2. Run all cells
3. Download `ocr_chunk_harmeme_{CHUNK_ID}.pkl` from `/kaggle/working`

Run 4 copies of this notebook **in parallel** (one per CHUNK_ID) to save time.


In [ ]:
import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pip_install("pytesseract", "Pillow", "tqdm")

# Tesseract binary — already on Kaggle; uncomment if missing
# subprocess.run(["apt-get", "install", "-y", "tesseract-ocr"], check=False)

print("✅ Packages ready.")


In [ ]:
import os, pickle
import pandas as pd
import pytesseract
from pathlib import Path
from PIL import Image
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor

print("✅ Imports done.")


In [ ]:
# ════════════════════════════════════════════════
#  ↓ ONLY CHANGE THIS  ↓
CHUNK_ID = 0   # set to 0, 1, 2, or 3
#  ↑ ONLY CHANGE THIS  ↑
# ════════════════════════════════════════════════

# chunk CSVs uploaded as dataset 'harmeme-chunks' after running Notebook 1
CHUNK_CSV  = Path(f"/kaggle/input/harmeme-chunks/harmeme_chunk_{CHUNK_ID}.csv")
CHUNK_SAVE = Path(f"/kaggle/working/ocr_chunk_harmeme_{CHUNK_ID}.pkl")
NUM_WORKERS = 2

assert CHUNK_CSV.exists(), f"❌ Chunk CSV not found: {CHUNK_CSV}\nMake sure 'harmeme-chunks' dataset is attached."

chunk_rows = pd.read_csv(CHUNK_CSV).to_dict("records")
print(f"✅ Chunk {CHUNK_ID}: {len(chunk_rows)} images to process")
print(f"   Sample row: {chunk_rows[0]}")


In [ ]:
# ── OCR function ─────────────────────────────────────────────────────────────
def get_ocr_words_and_boxes(img_path: str):
    """
    Returns (words, boxes) with boxes normalised to [0, 1000].
    Compatible with LiLT / LayoutLMv3 convention.
    """
    try:
        img  = Image.open(img_path).convert("RGB")
        W, H = img.size
        data = pytesseract.image_to_data(img, output_type=pytesseract.Output.DICT)

        words, boxes = [], []
        for i, word in enumerate(data["text"]):
            word = word.strip()
            if not word:
                continue
            x, y, w, h = data["left"][i], data["top"][i], data["width"][i], data["height"][i]
            x0 = max(0, min(1000, int(x / W * 1000)))
            y0 = max(0, min(1000, int(y / H * 1000)))
            x1 = max(0, min(1000, int((x + w) / W * 1000)))
            y1 = max(0, min(1000, int((y + h) / H * 1000)))
            words.append(word)
            boxes.append([x0, y0, x1, y1])

    except Exception:
        words, boxes = [], []

    # Always return at least one token
    if not words:
        words, boxes = ["[UNK]"], [[0, 0, 1000, 1000]]

    return words, boxes

print("✅ OCR function defined.")

# Quick test on first image
first = chunk_rows[0]
test_path = str(Path(first["img_root"]) / first["fname"])
if os.path.exists(test_path):
    tw, tb = get_ocr_words_and_boxes(test_path)
    print(f"   Test image : {first['fname']}  (source: {first['source']})")
    print(f"   Words      : {tw[:6]}")
    print(f"   Boxes      : {tb[:3]}")
else:
    print(f"⚠️  Test image not found: {test_path}")


In [ ]:
# ── Process chunk ─────────────────────────────────────────────────────────────
def process(row):
    fname    = row["fname"]
    img_path = str(Path(row["img_root"]) / fname)

    if not os.path.exists(img_path):
        return fname, {"words": ["[UNK]"], "boxes": [[0, 0, 1000, 1000]]}
    try:
        words, boxes = get_ocr_words_and_boxes(img_path)
    except Exception:
        words, boxes = ["[UNK]"], [[0, 0, 1000, 1000]]

    return fname, {"words": words, "boxes": boxes}


ocr_chunk = {}
with ThreadPoolExecutor(max_workers=NUM_WORKERS) as ex:
    for fname, entry in tqdm(
        ex.map(process, chunk_rows),
        total=len(chunk_rows),
        desc=f"OCR Chunk {CHUNK_ID}"
    ):
        ocr_chunk[fname] = entry

with open(CHUNK_SAVE, "wb") as f:
    pickle.dump(ocr_chunk, f)

print(f"\n✅ Chunk {CHUNK_ID} saved → {CHUNK_SAVE}")
print(f"   Entries : {len(ocr_chunk)}")
print(f"   Size    : {CHUNK_SAVE.stat().st_size/1e6:.1f} MB")
print(f"\nSample entries:")
for k, v in list(ocr_chunk.items())[:3]:
    print(f"  {k}: words={v['words'][:4]}  boxes={v['boxes'][:2]}")
